# RF / XGBoost: temporal-feature and diagnosis-embedding ablations

This time only code was added; no real patient experiment was run. All parameters, the development set and the outer split keep the original balanced definitions.

## Comparison design

- static_rf → temporal_rf; static_xgb → temporal_xgb: test the contribution of temporal features under different models.
- static_rf → burden_rf → embedding_rf; likewise for XGBoost: first control for the number of diagnoses, then judge the gain from the latent representation.
- embedding is also compared with the same-model static, reporting the total gain of the whole diagnosis extension.
- The same numeric LASSO screening is still used, so "changing the learner" and "changing the screening method" are not mixed into one comparison.

This entry point fits only standalone base models and does no internal stacking cross-fitting. Existing stacking predictions can be reused for the historical summary, and stacking is not retrained because of this.

## Actual procedure of the diagnosis embedding

1. Take only hospitalizations discharged before the index hospitalization: `prior.dischtime < index.admittime`; discharge diagnoses of the current hospitalization and future hospitalizations are not used.
2. ICD codes are stripped of decimal points and upper-cased, with ICD-9/10 labelled separately. Each exact code and its first-three-character category produce tokens.
   For example ICD-10 `E11.9` becomes `v10_E119` and `c10_E11`; ICD-9 uses `v9_` / `c9_`, so the two are never merged into the same code.
3. Each patient's historical diagnosis tokens are de-duplicated into one document. Because tokens are de-duplicated, the sublinear_tf of TF usually does not reflect repetition here; diagnosis frequency is represented separately by burden.
4. TF-IDF is fitted on the training fold with `min_df=10` (appearing in at least 10 training-patient documents); the validation set uses only the training vocabulary.
5. TruncatedSVD is fitted on the training fold, at most 32 dimensions; the validation set is only transformed and takes no part in learning the vocabulary, IDF or SVD.
6. Numeric static/temporal features go through within-fold missing-rate screening, imputation and LASSO; the four burden indicators and the embedding are appended directly after LASSO.
   Tree models have no extra StandardScaler; the first three burden counts get log1p. All of this is inherited from the existing Pipeline.

This is a statistical representation of structured diagnoses, not clinical text or a medical language model. The SVD dimensions cannot be named directly as particular diseases, and more dimensions are not guaranteed to improve prediction.
Patients with no historical diagnoses are kept with an empty document; an empty or single-term vocabulary has the existing fallback logic. Taking the first three characters of ICD-10 is a code grouping, not a complete mapping of the clinical disease hierarchy.

## Reproducibility and cache

Seeds 42–46, 10 outer folds; LASSO inner 5 folds/30 λ, 1-SE; missing-rate threshold 20%, keeping the original static white-cell differential exemption.
RF/XGBoost parameters are still read from the original config.yaml, without tuning. The fixed held-out labels take no part.

`--reuse-from ebf3f368bb8beb5c` verifies the old training functions, configuration, data, software environment and per-fold indices before copying the existing predictions.
Old results are kept and new experiments are written under a new signature; a mismatch raises an error and the checks are never bypassed. The Holm correction in the results is updated with the current comparison family.

Run in a terminal from the original extension directory:
```bash
bash balanced/run_extensions.sh full --experiments temporal_rf,temporal_xgb,embedding_rf,embedding_xgb --reuse-from ebf3f368bb8beb5c
```
Add `--check-only` at the end of the command to validate only, without training. The four experiments can also be split into two runs, temporal and embedding.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'dn_extensions/settings.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('start from the balanced directory or its notebooks subdirectory.')
sys.path.insert(0, str(ROOT))
print('project directory:', ROOT)


In [ ]:
# No training by default; set RUN_EXPERIMENT=True manually after you confirm.
RUN_EXPERIMENT = False
EXPERIMENTS = ['temporal_rf', 'temporal_xgb', 'embedding_rf', 'embedding_xgb']
REUSE_FROM = 'ebf3f368bb8beb5c'
if RUN_EXPERIMENT:
    from dn_extensions.evaluation import run_evaluation
    from dn_extensions.settings import profile_options
    output, summary, comparisons = run_evaluation(
        profile_options('full'), EXPERIMENTS, reuse_from=REUSE_FROM)
    display(summary)
    display(comparisons)
else:
    print('training not started; use the Bash command above to run it.')


In [ ]:
# Only show existing results, no retraining.
latest = ROOT / 'results/full/latest.txt'
if latest.exists():
    folder = latest.parent / latest.read_text().strip()
    table = pd.read_csv(folder / 'summary.csv')
    names = EXPERIMENTS + ['static_rf', 'static_xgb', 'burden_rf', 'burden_xgb']
    display(table[table.experiment.isin(names)])
    print('detailed results:', folder / 'report.md')
else:
    print('no full results yet.')


## Experiment status and conclusions (updated automatically)

Run mode: `full`; method: balanced fixed parameters; result directory: `results/full/27464ba98305edbc`.

LASSO 5 folds/30 λ, stacking 3 folds; static and extension models use the same simplified method. No folds of the strict version were mixed in.

The fixed held-out set took no part in this experiment. Tree-model extensions need no internal stacking cross-fitting; the original fixed parameters and within-training-fold preprocessing are unchanged.

The Holm correction covers the paired comparisons currently available; corrected p-values may change when experiments are added later.

```text

   experiment  n_outer_folds  AUROC_fold_mean  AUROC_fold_sd  AUPRC_fold_mean  Brier_fold_mean  calibration_slope_fold_mean  seconds_total  AUROC_pooled_mean  AUPRC_pooled_mean  Brier_pooled_mean  cal_intercept_pooled_mean  cal_slope_pooled_mean
    burden_rf             50          0.79724        0.02871          0.60343          0.15232                      1.22278       28.27988            0.79581            0.58966            0.15232                    0.00337                1.18404
   burden_xgb             50          0.79396        0.03022          0.60583          0.15265                      0.93979        6.26590            0.79309            0.59511            0.15264                    0.00653                0.91576
 embedding_rf             50          0.79965        0.02798          0.60840          0.15269                      1.39541       32.57728            0.79825            0.59542            0.15268                    0.00490                1.35224
embedding_xgb             50          0.79429        0.02760          0.60649          0.15232                      0.97897        9.29661            0.79331            0.59430            0.15231                   -0.00756                0.95188
    static_rf             50          0.79798        0.02809          0.60618          0.15177                      1.19841       15.22184            0.79671            0.59435            0.15177                    0.00179                1.16211
   static_xgb             50          0.79504        0.02965          0.60572          0.15250                      0.94083        4.61352            0.79408            0.59662            0.15249                    0.00766                0.91690
  temporal_rf             50          0.79741        0.02891          0.60594          0.15212                      1.24090       28.89751            0.79605            0.59470            0.15211                    0.00148                1.20217
 temporal_xgb             50          0.79284        0.02796          0.60391          0.15300                      0.94159        8.29193            0.79177            0.59437            0.15299                    0.01106                0.91792

```

Paired differences:

```text

   experiment  reference metric  mean_diff  sd_diff  se_corrected       lo      hi        t  df  p_value  k  r  p_holm_current_family
    static_rf  static_lr  AUROC   -0.00130  0.01457       0.00528 -0.01190 0.00931 -0.24552  49  0.80708 10  5                    1.0
    static_rf  static_lr  AUPRC   -0.00122  0.02900       0.01050 -0.02232 0.01989 -0.11569  49  0.90837 10  5                    1.0
    static_rf  static_lr  Brier    0.00085  0.00469       0.00170 -0.00256 0.00427  0.50207  49  0.61787 10  5                    1.0
   static_xgb  static_lr  AUROC   -0.00423  0.01409       0.00510 -0.01449 0.00602 -0.82968  49  0.41074 10  5                    1.0
   static_xgb  static_lr  AUPRC   -0.00167  0.02894       0.01048 -0.02273 0.01938 -0.15972  49  0.87376 10  5                    1.0
   static_xgb  static_lr  Brier    0.00158  0.00519       0.00188 -0.00220 0.00536  0.83926  49  0.40540 10  5                    1.0
  temporal_rf  static_rf  AUROC   -0.00057  0.00733       0.00265 -0.00591 0.00476 -0.21556  49  0.83022 10  5                    1.0
  temporal_rf  static_rf  AUPRC   -0.00024  0.01373       0.00497 -0.01023 0.00975 -0.04896  49  0.96115 10  5                    1.0
  temporal_rf  static_rf  Brier    0.00034  0.00196       0.00071 -0.00108 0.00177  0.48410  49  0.63047 10  5                    1.0
 temporal_xgb static_xgb  AUROC   -0.00220  0.01022       0.00370 -0.00964 0.00523 -0.59579  49  0.55405 10  5                    1.0
 temporal_xgb static_xgb  AUPRC   -0.00181  0.02169       0.00785 -0.01759 0.01397 -0.23090  49  0.81836 10  5                    1.0
 temporal_xgb static_xgb  Brier    0.00050  0.00389       0.00141 -0.00233 0.00334  0.35547  49  0.72376 10  5                    1.0
    burden_rf  static_rf  AUROC   -0.00074  0.00616       0.00223 -0.00522 0.00374 -0.33253  49  0.74091 10  5                    1.0
    burden_rf  static_rf  AUPRC   -0.00275  0.01287       0.00466 -0.01212 0.00661 -0.59075  49  0.55740 10  5                    1.0
    burden_rf  static_rf  Brier    0.00055  0.00173       0.00063 -0.00071 0.00181  0.87819  49  0.38412 10  5                    1.0
   burden_xgb static_xgb  AUROC   -0.00108  0.00676       0.00245 -0.00600 0.00384 -0.44121  49  0.66100 10  5                    1.0
   burden_xgb static_xgb  AUPRC    0.00010  0.01749       0.00633 -0.01263 0.01283  0.01602  49  0.98728 10  5                    1.0
   burden_xgb static_xgb  Brier    0.00015  0.00214       0.00077 -0.00140 0.00171  0.19740  49  0.84433 10  5                    1.0
 embedding_rf  burden_rf  AUROC    0.00241  0.00885       0.00320 -0.00403 0.00885  0.75282  49  0.45516 10  5                    1.0
 embedding_rf  burden_rf  AUPRC    0.00497  0.01607       0.00582 -0.00673 0.01666  0.85397  49  0.39728 10  5                    1.0
 embedding_rf  burden_rf  Brier    0.00037  0.00239       0.00087 -0.00137 0.00211  0.42331  49  0.67392 10  5                    1.0
embedding_xgb burden_xgb  AUROC    0.00033  0.01064       0.00385 -0.00741 0.00807  0.08550  49  0.93221 10  5                    1.0
embedding_xgb burden_xgb  AUPRC    0.00067  0.02335       0.00846 -0.01632 0.01766  0.07910  49  0.93728 10  5                    1.0
embedding_xgb burden_xgb  Brier   -0.00033  0.00395       0.00143 -0.00321 0.00254 -0.23263  49  0.81702 10  5                    1.0
 embedding_rf  static_rf  AUROC    0.00167  0.00963       0.00349 -0.00534 0.00868  0.47943  49  0.63377 10  5                    1.0
 embedding_rf  static_rf  AUPRC    0.00222  0.01939       0.00702 -0.01189 0.01632  0.31582  49  0.75348 10  5                    1.0
 embedding_rf  static_rf  Brier    0.00092  0.00260       0.00094 -0.00098 0.00281  0.97278  49  0.33544 10  5                    1.0
embedding_xgb static_xgb  AUROC   -0.00075  0.00988       0.00358 -0.00794 0.00644 -0.20974  49  0.83474 10  5                    1.0
embedding_xgb static_xgb  AUPRC    0.00077  0.02388       0.00865 -0.01661 0.01815  0.08909  49  0.92938 10  5                    1.0
embedding_xgb static_xgb  Brier   -0.00018  0.00389       0.00141 -0.00301 0.00265 -0.12793  49  0.89873 10  5                    1.0

```

- `static_rf` vs `static_lr`: ΔAUROC=-0.0013, corrected 95% CI [-0.0119, 0.0093]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `static_xgb` vs `static_lr`: ΔAUROC=-0.0042, corrected 95% CI [-0.0145, 0.0060]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `temporal_rf` vs `static_rf`: ΔAUROC=-0.0006, corrected 95% CI [-0.0059, 0.0048]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `temporal_xgb` vs `static_xgb`: ΔAUROC=-0.0022, corrected 95% CI [-0.0096, 0.0052]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `burden_rf` vs `static_rf`: ΔAUROC=-0.0007, corrected 95% CI [-0.0052, 0.0037]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `burden_xgb` vs `static_xgb`: ΔAUROC=-0.0011, corrected 95% CI [-0.0060, 0.0038]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `embedding_rf` vs `burden_rf`: ΔAUROC=0.0024, corrected 95% CI [-0.0040, 0.0089]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `embedding_xgb` vs `burden_xgb`: ΔAUROC=0.0003, corrected 95% CI [-0.0074, 0.0081]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `embedding_rf` vs `static_rf`: ΔAUROC=0.0017, corrected 95% CI [-0.0053, 0.0087]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `embedding_xgb` vs `static_xgb`: ΔAUROC=-0.0008, corrected 95% CI [-0.0079, 0.0064]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.